# 1. Imports

In [ ]:
import matplotlib.pyplot as plt 
import pandas
from pandas import DataFrame        
from IPython.display import Markdown, display 

# 2. Data Loading

In [ ]:
dataset: DataFrame = pandas.read_csv("data/caso_full.csv")

## 2.1 Schema Overview

In [ ]:
display(Markdown(f"**Amount of tuples**: {len(dataset)}"))

display(Markdown(f"**Column names and its types**:"))

columns_names_with_type: DataFrame = DataFrame(
    {
        "column_name": dataset.columns,
        "type": dataset.dtypes.values
    }    
)

columns_names_with_type

### Sample

In [ ]:
dataset.head(10)

### Memory Usage Before Optimization

In [ ]:
dataset_total_memory_usage = dataset.memory_usage(deep = True).sum() / (1024 ** 2)

display(
    Markdown(
        f"**Total memory usage:** {dataset_total_memory_usage:.2f} MB"
    )
)

# 3. Memory Optimization

In [ ]:
def optimize_dtypes(df: DataFrame) -> tuple[DataFrame, DataFrame]:
    conversions = {
        "city":                      ("category", None),
        "city_ibge_code":            ("Int32",    lambda c: c.round()),
        "date":                      ("datetime", None),
        "last_available_date":       ("datetime", None),
        "estimated_population":      ("Int32",    None),
        "estimated_population_2019": ("Int32",    None),
        "place_type":                ("category", lambda c: c.replace({"city": "C", "state": "S"})),
    }
    rows = []
    for col, (dtype, pre) in conversions.items():
        before = df[col].memory_usage(deep=True) / 1024**2
        if pre:
            df[col] = pre(df[col])
        if dtype == "datetime":
            df[col] = pandas.to_datetime(df[col])
        else:
            df[col] = df[col].astype(dtype)
        after = df[col].memory_usage(deep=True) / 1024**2
        rows.append({
            "column":      col,
            "before_MB":   round(before, 3),
            "after_MB":    round(after, 3),
            "reduction_%": round((before - after) / before * 100, 1),
        })
    return df, DataFrame(rows)

dataset, memory_report = optimize_dtypes(dataset)
memory_report

## Memory Usage After Optimization

In [ ]:
dataset_total_memory_usage_after_optimization = dataset.memory_usage(deep = True).sum() / (1024 ** 2)

display(
    Markdown(
        f"**Total memory usage:** {dataset_total_memory_usage_after_optimization:.2f} MB"
    )
)

In [ ]:
before = dataset_total_memory_usage
after  = dataset_total_memory_usage_after_optimization
saved  = before - after
reduction = (saved / before) * 100

labels = ["Before Optimization", "After Optimization"]
values = [before, after]

plt.figure(figsize=(8, 5))

bars = plt.bar(labels, values)

plt.title("Dataset Memory Usage Optimization")
plt.ylabel("Memory Usage (MB)")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.0f} MB",
        ha="center",
        va="bottom",
        fontsize=11,
    )

plt.text(
    0.5,
    max(values) * 0.6,
    f"Memory reduced by {reduction:.2f}%\n({saved:.0f} MB saved)",
    ha="center",
    fontsize=12,
)

plt.show()

# 4. Missing Values Analysis

## Summary

In [ ]:
display(
    Markdown(f"**Total amount of missing values**: {dataset.isna().sum().sum()}")
)

## Per-Column Detail

In [ ]:
empty_data_reckoning_rendered: DataFrame = DataFrame(dataset.isna().sum().sort_values(ascending = True))
empty_data_reckoning_rendered

### city

In [ ]:
missing_city_mask = (dataset["city"].isna()) | (dataset["city"] == "Importados/Indefinidos")

empty_city_values = dataset.loc[
    missing_city_mask,
    ["city", "state", "city_ibge_code"]
]

empty_city_values

In [ ]:
missing_city_rate: float = len(empty_city_values) / len(dataset)

display(
    Markdown(
        f"**Missing city rate:** {missing_city_rate:.2f}%"
    )
)

display(
    Markdown(
        "**So... drop it!**" if missing_city_rate < 0.25 else "**Consider imputing values**"
    )
)

### last_available_confirmed_per_100k_inhabitants

In [ ]:
missing_last_available_per_100k_inhabitants_mask = dataset["last_available_confirmed_per_100k_inhabitants"].isna()

empty_values_last_100k = dataset.loc[
    missing_last_available_per_100k_inhabitants_mask,
    ["last_available_confirmed_per_100k_inhabitants", "city", "state", "city_ibge_code"]
]

empty_values_last_100k

### estimated_population_2019

In [ ]:
missing_estimated_population_2019_mask = (dataset["estimated_population_2019"].isna() | dataset["estimated_population_2019"] == "<NA>")

estimated_population = dataset.loc[
    missing_estimated_population_2019_mask,
    ["estimated_population", "estimated_population_2019", "city", "state", "city_ibge_code"]
]

estimated_population

### estimated_population

In [ ]:
missing_estimated_population_mask = (dataset["estimated_population"].isna())

missing_estimated_population_lines = dataset.loc[
    missing_estimated_population_mask,
    ["estimated_population", "city", "state", "city_ibge_code"]
]

missing_estimated_population_lines

### city_ibge_code

In [ ]:
missing_city_ibge_mask = (dataset["city_ibge_code"].isna()) | (dataset["city_ibge_code"] == "<NA>")

missing_city_ibge_list = dataset.loc[
    missing_city_ibge_mask,
    ["city", "state", "city_ibge_code"]
]

missing_city_ibge_list

In [ ]:
dataset.loc[
    dataset["last_available_confirmed_per_100k_inhabitants"].isna(),
    "estimated_population"
].describe()

# 5. Data Integrity

## 5.1 Repeated Records (is_repeated)

In [ ]:
repeated_counts = dataset["is_repeated"].value_counts()

display(Markdown(f"**Repeated records:** {repeated_counts.get(True, 0):,}"))
display(Markdown(f"**Non-repeated records:** {repeated_counts.get(False, 0):,}"))

dataset[dataset["is_repeated"]].head(5)

**Decision:** `is_repeated` days were interpolated rather than zeroed out because this flag indicates that the state health secretariat did not publish a bulletin that day — not that no cases occurred. Keeping zero values would bias the LSTM into learning spurious dips in the series.

## 5.2 Latest Record Flag (is_last)

In [ ]:
is_last_counts = dataset["is_last"].value_counts()
total = len(dataset)

display(Markdown(f"**is_last = True:** {is_last_counts.get(True, 0):,} ({is_last_counts.get(True, 0) / total * 100:.1f}%)"))
display(Markdown(f"**is_last = False:** {is_last_counts.get(False, 0):,} ({is_last_counts.get(False, 0) / total * 100:.1f}%)"))

dataset[dataset["is_last"]].head(5)

**Decision:** `is_last` was filtered to `True` to retain only the most recent, corrected version of each record, avoiding double-counting of cases that were later revised.

## 5.3 Negative Values in new_confirmed

In [ ]:
negatives = dataset[dataset["new_confirmed"] < 0]

display(Markdown(f"**Negative new_confirmed records:** {len(negatives):,}"))

if len(negatives) > 0:
    display(negatives[["date", "state", "city", "new_confirmed"]].head(10))

**Decision:** `new_confirmed` was clipped at zero because negative values do not indicate a decrease in cases; they reflect reclassifications between municipalities by the secretariat. Clipping preserves series integrity without introducing artificial drops.

# 6. Data Coverage

In [ ]:
state_level = dataset[
    (dataset["place_type"] == "S") &
    (dataset["is_last"] == True)
]

coverage = (
    state_level.groupby("state")["date"]
    .agg(start="min", end="max", days="count")
    .sort_values("days")
)

display(Markdown(f"**Date range:** {state_level['date'].min().date()} → {state_level['date'].max().date()}"))
display(Markdown(f"**States present:** {state_level['state'].nunique()} / 27"))
coverage

# 7. Data Cleaning Pipeline

`place_type` was filtered to state level because state-aggregated series are 
less noisy than city-level data and provide denser, more consistent coverage.

`is_repeated` days were interpolated rather than zeroed out because this flag 
indicates that the state health secretariat did not publish a bulletin that day 
— not that no cases occurred. Keeping zero values would bias the LSTM into 
learning spurious dips in the series.

`new_confirmed` was clipped at zero because negative values do not indicate a 
decrease in cases; they reflect reclassifications between municipalities by the 
secretariat. Clipping preserves series integrity without introducing artificial 
drops.

`is_last` was filtered to True to retain only the most recent, corrected version 
of each record, avoiding double-counting of cases that were later revised.


In [ ]:
df = dataset[
    (dataset["place_type"] == "S") &
    (dataset["is_last"] == True)
].copy().sort_values(["state", "date"])

def clean_state_series(group):
    group = group.copy()
    group["new_confirmed"] = (
        group["new_confirmed"]
        .where(~group["is_repeated"])
        .interpolate(method="linear")
    )
    group["new_confirmed"] = group["new_confirmed"].clip(lower=0)
    return group

df = df.groupby("state", group_keys=False).apply(clean_state_series)

display(Markdown(f"**Records after cleaning:** {len(df):,}"))
display(Markdown(f"**Date range:** {df['date'].min().date()} → {df['date'].max().date()}"))
display(Markdown(f"**States:** {df['state'].nunique()}"))
df.head()